In [1]:
!pip install -q qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 10.8 MB/s eta 0:00:0000:01


In [ ]:
"""Đánh giá chất lượng retrieve luật khi QUERY = case_query + chunks evidence (agent_v4),
bằng ĐÚNG bộ metric trong `evaluate-retrieval-bge-m3.ipynb`.

KHÔNG so sánh biến thể (không còn baseline case_query trơn / legal_query / hybrid /
rerank) — chỉ chấm MỘT thứ: dense bge-m3 search với query đã nối thêm evidence chunk
tương ứng từng case trong agent_v4_results.json (cùng nguồn evidence dùng ở luồng chính
fullchunks). KHÔNG sinh file retrieval cho luồng judge chính — chỉ để xem số liệu.

CHỈ CHẤM ĐƯỢC TRÊN PUBLIC (cần gold `related_law_provisions`).

Metric (đúng notebook, K=1,3,5,10): Precision, Recall macro/micro corpus-conditioned,
Recall end-to-end, Hit Rate, MRR, nDCG, MAP — khớp (law_id + article_no) version-aware.
BONUS F2 = metric CHÍNH THỨC ALQAC (ưu tiên recall gấp đôi precision): F2=5PR/(4P+R).

CHẠY TRÊN KAGGLE (bge-m3 cần GPU; Qdrant collection đã dựng bởi embed-model-retrieval-bge-m3.ipynb):
  - Bật GPU. Kaggle Secrets: QDRANT_URL, QDRANT_KEY (như notebook embedding).
  - Upload: corpus_law_pub.json, ALQAC2026_public_test.json, agent_v4_results.json.
  - Dán cả file này vào 1 cell rồi Run, hoặc: !python eval_retrieval_bge.py
KHÔNG chạy được ở máy local (không có GPU/bge-m3/Qdrant) — đây là script cho Kaggle.
"""
from __future__ import annotations

import glob
import json
import math
import os
import re
import unicodedata

import numpy as np
import pandas as pd

MODEL_ID = "BAAI/bge-m3"
COLLECTION = "laws_bge_m3_v2_correct_pooling"
KS = (1, 3, 5, 10)
TOP_K = 10
MAX_LENGTH = 8192

# Văn bản THỦ TỤC — semantic KHÔNG bắt được (hệ thật tiêm bằng kênh rule-based riêng).
# Chế độ substantive-only bỏ mấy điều này khỏi gold để chấm ĐÚNG phần semantic nhắm tới.
PROCEDURAL_LAW_IDS = {"92/2015/QH13", "93/2015/QH13", "326/2016/UBTVQH14", "26/2008/QH12"}

# Pool THỦ TỤC lõi (copy từ tools/law_procedural.py để script chạy STANDALONE trên Kaggle) —
# calibrate từ tần suất gold thủ tục trên public, tiêm cho mọi vụ dân sự sơ thẩm.
# (law_id, article_no). Dùng cho chế độ GHÉP: semantic(nội dung) ∪ rule(thủ tục).
CORE_PROCEDURAL = [
    ("92/2015/QH13", 26), ("92/2015/QH13", 147), ("92/2015/QH13", 35),
    ("92/2015/QH13", 39), ("92/2015/QH13", 273), ("92/2015/QH13", 157),
    ("92/2015/QH13", 165), ("326/2016/UBTVQH14", 12), ("326/2016/UBTVQH14", 26),
    ("92/2015/QH13", 271), ("92/2015/QH13", 266),
]
K_SUB_SWEEP = (5, 8, 10, 12, 15, 20)


# ---------------------------------------------------------------------------
# 0. Nạp secrets + file (Kaggle Secrets hoặc biến môi trường).
# ---------------------------------------------------------------------------
def _get_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ[name]


def _find(name: str) -> str:
    cands = [name, f"/kaggle/working/{name}"] + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for path in cands:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Không thấy {name} (upload vào Kaggle input hoặc /kaggle/working).")


def load_json(path: str):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ---------------------------------------------------------------------------
# 1. Gold parsing version-aware — COPY NGUYÊN từ evaluate-retrieval-bge-m3.ipynb
#    để số liệu SO SÁNH ĐƯỢC trực tiếp với baseline của họ (cùng định nghĩa gold).
# ---------------------------------------------------------------------------
def normalize_text(value) -> str:
    value = unicodedata.normalize("NFD", str(value or ""))
    value = "".join(ch for ch in value if unicodedata.category(ch) != "Mn")
    value = value.replace("đ", "d").replace("Đ", "D").lower()
    return re.sub(r"\s+", " ", value).strip()


CANONICAL_TO_CORPUS_LAW_ID = {
    "BLTTDS_2015": "92/2015/QH13", "BLDS_2015": "91/2015/QH13", "HNGD_2014": "52/2014/QH13",
    "DAT_DAI_2013": "45/2013/QH13", "THADS_2008": "26/2008/QH12", "HO_TICH_2014": "60/2014/QH13",
    "TCTD_2010": "47/2010/QH12", "XAY_DUNG_2014": "50/2014/QH13", "TTHC_2015": "93/2015/QH13",
    "KHIEU_NAI_2011": "02/2011/QH13", "KDBDS_2014": "66/2014/QH13",
    "NQ326_2016": "326/2016/UBTVQH14", "ND37_2015": "37/2015/NĐ-CP",
}


def first_explicit_year(text, *years):
    return next((year for year in years if str(year) in text), None)


def canonical_gold_law(title: str) -> str:
    value = normalize_text(title)
    if "bo luat to tung dan su" in value or value == "bo luat to tung":
        return f"BLTTDS_{first_explicit_year(value, 2015) or 2015}"
    if "bo luat dan su" in value:
        return f"BLDS_{first_explicit_year(value, 1995, 2005, 2015) or 2015}"
    if "luat hon nhan va gia dinh" in value or "luat hon nhan gia dinh" in value:
        return f"HNGD_{first_explicit_year(value, 2000, 2014) or 2014}"
    if "luat dat dai" in value:
        return f"DAT_DAI_{first_explicit_year(value, 1987, 1993, 2003, 2013) or 2013}"
    if "luat thi hanh an dan su" in value:
        return "THADS_2008"
    if "luat ho tich" in value:
        return "HO_TICH_2014"
    if "luat cac to chuc tin dung" in value:
        return "TCTD_2010"
    if "luat xay dung" in value:
        return f"XAY_DUNG_{first_explicit_year(value, 2003, 2014) or 2014}"
    if "luat to tung hanh chinh" in value:
        return "TTHC_2015"
    if "luat khieu nai, to cao" in value or "luat khieu nai to cao" in value:
        return f"KHIEU_NAI_TO_CAO_{first_explicit_year(value, 1998, 2006) or 'OLD'}"
    if "luat khieu nai" in value:
        return "KHIEU_NAI_2011"
    if "luat kinh doanh bat dong san" in value:
        return f"KDBDS_{first_explicit_year(value, 2006, 2014) or 2014}"
    if "326/2016" in value or re.search(r"nghi quyet (so )?:? ?326\b", value):
        return "NQ326_2016"
    if "37/2015/nd-cp" in value:
        return "ND37_2015"
    if "181/2004/nd-cp" in value:
        return "ND181_2004"
    if "phap lenh" in value and "an phi" in value:
        return "PHAP_LENH_AN_PHI_2009"
    return re.sub(r"ngay .*$", "", value).rstrip(";, .")


def build_parse_gold(corpus_by_key):
    def parse_gold(row):
        all_gold, available_gold, unavailable_gold = set(), set(), set()
        for raw_line in str(row.get("related_law_provisions", "")).splitlines():
            line = raw_line.strip()
            if not line:
                continue
            article_numbers = [int(x) for x in re.findall(r"\bdieu\s+(\d+)", normalize_text(line))]
            if not article_numbers:
                continue
            title = line.split("|", 1)[0].strip()
            canonical = canonical_gold_law(title)
            corpus_law_id = CANONICAL_TO_CORPUS_LAW_ID.get(canonical)
            for article_no in article_numbers:
                all_gold.add((canonical, article_no))
                exact_key = (corpus_law_id, article_no) if corpus_law_id else None
                if exact_key and exact_key in corpus_by_key:
                    available_gold.add(exact_key)
                else:
                    unavailable_gold.add((canonical, article_no))
        return {"all": all_gold, "available": available_gold, "unavailable": unavailable_gold}
    return parse_gold


# ---------------------------------------------------------------------------
# 2. Metric per-case — COPY NGUYÊN evaluate_case + thêm F2 (ALQAC).
# ---------------------------------------------------------------------------
def evaluate_case(gold_row, retrieval, parse_gold, substantive_only=False):
    case_id = gold_row["case_id"]
    gold = parse_gold(gold_row)
    available = gold["available"]
    if substantive_only:
        available = {key for key in available if key[0] not in PROCEDURAL_LAW_IDS}
    ranked_keys = [(str(it["law_id"]), int(it["article_no"])) for it in retrieval[case_id]]
    result = {"case_id": case_id, "gold_all": len(gold["all"]),
              "gold_available": len(available), "gold_unavailable": len(gold["unavailable"])}
    for k in KS:
        relevance = [key in available for key in ranked_keys[:k]]
        hits = sum(relevance)
        first_hit = next((i for i, flag in enumerate(relevance, 1) if flag), None)
        dcg = sum(1 / math.log2(i + 1) for i, flag in enumerate(relevance, 1) if flag)
        idcg = sum(1 / math.log2(i + 1) for i in range(1, min(len(available), k) + 1))
        running, precision_sum = 0, 0.0
        for i, flag in enumerate(relevance, 1):
            if flag:
                running += 1
                precision_sum += running / i
        p = hits / k
        r = hits / len(available) if available else None
        result[f"hits@{k}"] = hits
        result[f"precision@{k}"] = p
        result[f"recall_available@{k}"] = r
        result[f"recall_end_to_end@{k}"] = hits / len(gold["all"]) if gold["all"] else None
        result[f"hit@{k}"] = int(hits > 0)
        result[f"rr@{k}"] = 1 / first_hit if first_hit else 0.0
        result[f"ndcg@{k}"] = dcg / idcg if idcg else None
        result[f"ap@{k}"] = precision_sum / len(available) if available else None
        # F2 (ALQAC chính thức): ưu tiên recall gấp đôi precision.
        result[f"f2@{k}"] = (5 * p * r / (4 * p + r)) if (r is not None and (4 * p + r) > 0) else None
    return result


def aggregate(gold_rows, retrieval, parse_gold, substantive_only=False) -> dict:
    df = pd.DataFrame(
        evaluate_case(row, retrieval, parse_gold, substantive_only=substantive_only)
        for row in gold_rows
    )
    total_gold = int(df["gold_all"].sum())
    total_avail = int(df["gold_available"].sum())
    out = {"_hit_any@10": int((df["hit@10"] == 1).sum()), "_n": len(df)}
    for k in KS:
        hits = int(df[f"hits@{k}"].sum())
        out[k] = {
            "Precision": hits / (len(df) * k),
            "Recall_macro_corpus": df[f"recall_available@{k}"].mean(),
            "Recall_micro_corpus": hits / total_avail if total_avail else None,
            "Recall_macro_e2e": df[f"recall_end_to_end@{k}"].mean(),
            "HitRate": df[f"hit@{k}"].mean(),
            "MRR": df[f"rr@{k}"].mean(),
            "nDCG": df[f"ndcg@{k}"].mean(),
            "MAP": df[f"ap@{k}"].mean(),
            "F2": df[f"f2@{k}"].mean(),
        }
    return {"metrics": out, "per_case": df, "total_gold": total_gold, "total_available": total_avail}


# ---------------------------------------------------------------------------
# 3. Main: query = case_query + chunks (agent_v4), dense bge-m3 search, chấm điểm.
# ---------------------------------------------------------------------------
def _clean_case_query(row) -> str:
    # KHONG bo "Agent du doan..." hay collapse whitespace -- khop DUNG production
    # (only_query/fullchunks dung case['case_query'].strip() truc tiep, khong qua regex nao).
    return str(row.get("case_query", "")).strip()


def main():
    import importlib.util
    import subprocess
    import sys
    if importlib.util.find_spec("qdrant_client") is None:
        print("Cài qdrant-client ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qdrant-client"], check=True)
    if importlib.util.find_spec("sentence_transformers") is None:
        print("Cài sentence-transformers (không -U) ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)

    import torch
    from qdrant_client import QdrantClient
    from sentence_transformers import SentenceTransformer

    assert torch.cuda.is_available(), "Bật GPU Kaggle."
    qc = QdrantClient(url=_get_secret("QDRANT_URL"), api_key=_get_secret("QDRANT_KEY"), timeout=300)
    stored_count = qc.count(COLLECTION).count
    assert stored_count == 3352, f"Qdrant thieu diem: {stored_count}/3352 -- chay lai embed-model-retrieval-bge-m3.ipynb truoc."
    print("Qdrant points:", stored_count)

    corpus = load_json(_find("corpus_law_pub.json"))
    gold_rows = load_json(_find("ALQAC2026_public_test.json"))
    agent_v4 = load_json(_find("agent_v4_results.json"))

    corpus_by_key = {}
    for law in corpus:
        for article_no, article in enumerate(law["content"], 1):
            corpus_by_key[(str(law["law_id"]), article_no)] = {
                "aid": int(article["aid"]), "text": str(article.get("content_Article", "")),
            }
    parse_gold = build_parse_gold(corpus_by_key)

    # Chunks evidence (agent_v4) -- CUNG NGUON evidence_details[].text dung o luong chinh
    # fullchunks (build_user_prompt facts_block); noi vao case_query truoc khi embed/search.
    MAX_EVIDENCE_CHARS = 8000

    def _case_evidence_text(record, max_chars=MAX_EVIDENCE_CHARS):
        seen, parts, total = set(), [], 0
        for e in record.get("evidence_details", []):
            chunk_id = e.get("chunk_id")
            txt = str(e.get("text", "")).strip()
            if not txt or chunk_id in seen:
                continue
            seen.add(chunk_id)
            if total + len(txt) + 1 > max_chars:
                break
            parts.append(txt)
            total += len(txt) + 1
        return "\n".join(parts)

    evidence_by_case = {r["case_id"]: _case_evidence_text(r) for r in agent_v4}
    _missing_ev = [row["case_id"] for row in gold_rows if not evidence_by_case.get(row["case_id"])]
    if _missing_ev:
        print("CANH BAO: thieu evidence agent_v4 cho case:", _missing_ev)

    def _query_plus_chunks(row):
        cq = _clean_case_query(row)
        chunks = evidence_by_case.get(row["case_id"], "")
        return (cq + "\n\n" + chunks).strip() if chunks else cq

    model = SentenceTransformer(MODEL_ID, device="cuda")
    model.max_seq_length = MAX_LENGTH

    def _dense(query, limit):
        qv = model.encode([query or " "], normalize_embeddings=True, convert_to_numpy=True)[0]
        hits = qc.query_points(COLLECTION, query=qv.tolist(), limit=limit, with_payload=True).points
        return [(str(h.payload["law_id"]), int(h.payload["article_no"]), int(h.payload["aid"])) for h in hits]

    def _items(keys3):
        return [{"rank": i + 1, "law_id": k[0], "article_no": k[1], "aid": k[2], "score": 0.0}
                for i, k in enumerate(keys3)]

    NAME = "case_query+chunks (agent_v4, bge-m3 dense)"
    retrieval = {row["case_id"]: _items(_dense(_query_plus_chunks(row), max(20, TOP_K))) for row in gold_rows}
    print(f"✓ retrieved: {NAME}")

    main_metrics = ["Precision", "Recall_macro_corpus", "HitRate", "MRR", "nDCG", "MAP", "F2"]
    pd.set_option("display.width", 200)
    pd.set_option("display.max_colwidth", 40)
    saved = {}

    def report(substantive_only: bool, title: str):
        result = aggregate(gold_rows, retrieval, parse_gold, substantive_only=substantive_only)
        n_gold = result["total_available"]
        table = pd.DataFrame({NAME: {m: result["metrics"][TOP_K][m] for m in main_metrics}}).T
        print(f"\n{title}  (tổng gold chấm = {n_gold} điều)")
        print("-" * 92)
        print((table * 100).round(2).to_string())
        saved[title] = {NAME: result["metrics"]}
        return result

    print("\n" + "#" * 92)
    print("# PHẦN 1 — RETRIEVAL THUẦN TÚY (KHÔNG nhồi/tiêm gì) — số liệu THẬT của case_query+chunks")
    print("#" * 92)
    report(False, "Bảng 1a. FULL GOLD (gồm cả điều thủ tục lẫn nội dung)")
    report(True, "Bảng 1b. SUBSTANTIVE-ONLY (chỉ điều NỘI DUNG — đúng phần semantic nhắm tới)")

    # ---- GHÉP: semantic(nội dung) ∪ rule(thủ tục), chấm FULL gold + F2 (metric ALQAC) ----
    proc_keys_ordered = [k for k in dict.fromkeys(CORE_PROCEDURAL) if k in corpus_by_key]
    proc_keys = set(proc_keys_ordered)
    golds_full = {row["case_id"]: parse_gold(row)["available"] for row in gold_rows}
    scored = [row["case_id"] for row in gold_rows if golds_full[row["case_id"]]]

    def f2(p, r):
        return (5 * p * r / (4 * p + r)) if (4 * p + r) > 0 else 0.0

    def eval_combined(add_proc, k_sub):
        P = R = F = 0.0
        for cid in scored:
            ranked = [(it["law_id"], it["article_no"]) for it in retrieval[cid]]
            sub = set(ranked[:k_sub])
            got = (sub | proc_keys) if add_proc else sub
            gold = golds_full[cid]
            hit = len(got & gold)
            p = hit / len(got) if got else 0.0
            r = hit / len(gold)
            P += p; R += r; F += f2(p, r)
        n = len(scored)
        return {"precision": P / n, "recall": R / n, "f2": F / n, "k_sub": k_sub}

    print("\n" + "#" * 92)
    print("# PHẦN 2 — KẾT HỢP semantic + TIÊM CỨNG 11 điều thủ tục (CORE_PROCEDURAL) vào MỌI case")
    print("#" * 92)
    print("LƯU Ý ĐỌC SỐ: các bảng dưới đây KHÔNG đo chất lượng retrieval của case_query+chunks.")
    print("11 điều thủ tục (thẩm quyền, án phí, thời hiệu...) được CỘNG CỨNG vào MỌI case, bất kể")
    print("case đó có liên quan hay không. Vì luật thủ tục gần như luôn có mặt trong gold của mọi")
    print("vụ dân sự sơ thẩm, việc tiêm cứng sẽ 'trúng' thường xuyên -> HitRate/F2/MRR nhảy vọt")
    print("DÙ retrieval semantic không hề thay đổi. Đây là mô phỏng kỹ thuật rule-injection THẬT")
    print("dùng ở luồng production (feed law_evidence cuối), KHÔNG phải bằng chứng retrieval giỏi.")
    print("Muốn biết case_query+chunks tự nó tốt tới đâu -> xem PHẦN 1 ở trên, không xem phần này.")

    print("\nSweep k_sub (số điều semantic giữ lại trước khi cộng rule) theo F2 (metric ALQAC):")
    print("-" * 92)
    print(f"{'variant':42s}{'Prec':>8}{'Recall':>9}{'F2':>8}{'k_sub*':>8}")
    combo = {}
    for add_proc, tag in ((False, "  (chỉ semantic, để đối chiếu)"), (True, "  (semantic + rule)")):
        best = max((eval_combined(add_proc, k) for k in K_SUB_SWEEP), key=lambda d: d["f2"])
        combo[NAME + (" + rule" if add_proc else "")] = best
        label = (NAME[:20] + tag)[:42]
        print(f"{label:42s}{best['precision']*100:7.2f}%"
              f"{best['recall']*100:8.2f}%{best['f2']*100:7.2f}%{best['k_sub']:>7}")

    # ---- Bảng 3: FULL bộ metric cho GHÉP semantic + rule, tại k_sub tối ưu F2 ----
    def _safe_mean(values):
        vals = [v for v in values if v is not None and math.isfinite(v)]
        return sum(vals) / len(vals) if vals else None

    def _rank_metrics(ranked_keys, gold_available):
        k = len(ranked_keys)
        relevance = [key in gold_available for key in ranked_keys]
        hits = sum(relevance)
        first_hit = next((i for i, flag in enumerate(relevance, 1) if flag), None)
        dcg = sum(1 / math.log2(i + 1) for i, flag in enumerate(relevance, 1) if flag)
        idcg = sum(1 / math.log2(i + 1) for i in range(1, min(len(gold_available), k) + 1)) if k else 0.0
        running, precision_sum = 0, 0.0
        for i, flag in enumerate(relevance, 1):
            if flag:
                running += 1
                precision_sum += running / i
        p = hits / k if k else 0.0
        r = hits / len(gold_available) if gold_available else None
        return {
            "Precision": p, "Recall_macro_corpus": r, "HitRate": int(hits > 0),
            "MRR": 1 / first_hit if first_hit else 0.0,
            "nDCG": (dcg / idcg) if idcg else None,
            "MAP": (precision_sum / len(gold_available)) if gold_available else None,
            "F2": f2(p, r) if r is not None else None,
        }

    k_sub = combo[NAME + " + rule"]["k_sub"]
    per_case_with_rule = []
    per_case_semantic_only = []
    n_hit_from_rule_only = 0
    for cid in scored:
        sem_ranked = [(it["law_id"], it["article_no"]) for it in retrieval[cid]]
        sem_top = sem_ranked[:k_sub]
        combined_ranked = list(dict.fromkeys(sem_top + proc_keys_ordered))
        gold_here = golds_full[cid]
        per_case_with_rule.append(_rank_metrics(combined_ranked, gold_here))
        per_case_semantic_only.append(_rank_metrics(sem_top, gold_here))
        # case ma cai hit CHI den tu rule tiem, khong den tu semantic top-k_sub
        hit_sem = bool(set(sem_top) & gold_here)
        hit_combined = bool(set(combined_ranked) & gold_here)
        if hit_combined and not hit_sem:
            n_hit_from_rule_only += 1
    full_combo_row = {m: _safe_mean([row[m] for row in per_case_with_rule]) for m in main_metrics}
    semantic_only_at_ksub_row = {m: _safe_mean([row[m] for row in per_case_semantic_only]) for m in main_metrics}

    print(f"\nBảng 2. FULL METRIC tại k_sub={k_sub} (tối ưu F2) — so sánh CÓ rule vs KHÔNG rule cùng k_sub")
    print("-" * 92)
    compare_table = pd.DataFrame({
        f"chỉ semantic (top-{k_sub}, không tiêm)": semantic_only_at_ksub_row,
        f"semantic (top-{k_sub}) + tiêm 11 rule": full_combo_row,
    }).T
    print((compare_table * 100).round(2).to_string())
    print(f"\n-> Trong {len(scored)} case chấm được, {n_hit_from_rule_only} case CHỈ trúng gold NHỜ 11 điều")
    print(f"   tiêm cứng (semantic top-{k_sub} không tự trúng case đó) — đây là phần 'ảo' do rule mang lại,")
    print("   không phải do case_query+chunks retrieve tốt hơn.")

    # ---- BẢNG 4: 4 METRIC @TOP_K khi NHỒI N điều thủ tục vào ĐẦU top-TOP_K (procedural-FIRST) ----
    def _at_top_k(ranked, gold):
        top = ranked[:TOP_K]
        hit = [k in gold for k in top]
        nh = sum(hit)
        first = next((i for i, h in enumerate(hit, 1) if h), None)
        return {
            "Recall_macro_corpus": nh / len(gold) if gold else 0.0,
            "HitRate": 1.0 if nh else 0.0,
            "MRR": 1.0 / first if first else 0.0,
            "Precision": nh / TOP_K,
        }

    print(f"\nBảng 3. Sweep N điều thủ tục nhồi vào ĐẦU top-{TOP_K} (procedural-FIRST), so N=0 (không tiêm) vs N tối ưu")
    print("-" * 92)
    at_topk_metrics = ["Recall_macro_corpus", "HitRate", "MRR", "Precision"]
    rows_by_n = {}
    for n_proc in range(0, TOP_K + 1):
        agg = {m: 0.0 for m in at_topk_metrics}
        for cid in scored:
            sem = [(it["law_id"], it["article_no"]) for it in retrieval[cid]]
            ranked = list(dict.fromkeys(proc_keys_ordered[:n_proc] + sem))
            m = _at_top_k(ranked, golds_full[cid])
            for k in agg:
                agg[k] += m[k]
        rows_by_n[n_proc] = {k: agg[k] / len(scored) for k in agg}
    best_n = max(rows_by_n, key=lambda n: sum(rows_by_n[n].values()))
    at_topk_best = {**rows_by_n[best_n], "N_proc*": best_n}
    n0_vs_best = pd.DataFrame({
        "N_proc=0 (không tiêm, retrieval thuần)": rows_by_n[0],
        f"N_proc={best_n} (tối ưu, có tiêm)": rows_by_n[best_n],
    }).T
    print((n0_vs_best * 100).round(2).to_string())
    print(f"\n-> Chênh lệch giữa 2 dòng trên GẦN NHƯ TOÀN BỘ đến từ việc tiêm {best_n} điều thủ tục vào đầu,")
    print("   không phải từ chất lượng semantic retrieval (2 dòng dùng CHUNG kết quả semantic retrieval).")

    print("\n" + "#" * 92)
    print("# TÓM TẮT — đọc đúng: PHẦN 1 = retrieval thật; PHẦN 2 = retrieval + tiêm rule (kỹ thuật khác)")
    print("#" * 92)
    full_gold_metrics = saved["Bảng 1a. FULL GOLD (gồm cả điều thủ tục lẫn nội dung)"][NAME][TOP_K]
    substantive_metrics = saved["Bảng 1b. SUBSTANTIVE-ONLY (chỉ điều NỘI DUNG — đúng phần semantic nhắm tới)"][NAME][TOP_K]
    summary_table = pd.DataFrame({
        "1a. retrieval thuần, full gold": {m: full_gold_metrics[m] for m in main_metrics},
        "1b. retrieval thuần, substantive-only": {m: substantive_metrics[m] for m in main_metrics},
        "2.  retrieval + tiêm rule (combo tối ưu)": full_combo_row,
    }).T
    print((summary_table * 100).round(2).to_string())

    # Lưu artifact.
    out = {
        **saved,
        "combined_f2": combo,
        "combined_full_metrics": {"semantic_only_at_k_sub": semantic_only_at_ksub_row,
                                   "semantic_plus_rule": full_combo_row,
                                   "n_hit_from_rule_only": n_hit_from_rule_only, "k_sub": k_sub},
        "at_top_k_procedural_first": {"n0_no_injection": rows_by_n[0], "best": at_topk_best},
    }
    with open("retrieval_eval_case_query_plus_chunks.json", "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2, default=lambda x: None if x is None else float(x))
    print("\n[saved] retrieval_eval_case_query_plus_chunks.json")


if __name__ == "__main__":
    main()


In [4]:
"""Sinh file retrieval top-10 (đúng cấu trúc data/retrieval_top10_bge_m3.json) để THAY
cho bản bge-m3 THÔ, làm ĐẦU VÀO cho luồng chính (judge phán ai thắng kiện).

Vì file này FEED CHO JUDGE (phán thắng/thua): chỉ lấy điều NỘI DUNG (semantic tốt nhất
= legal_query + hybrid dense+BM25). KHÔNG nhồi điều thủ tục (án phí/thẩm quyền) — chúng
vô nghĩa với việc phán và sẽ bỏ đói judge. (Điều thủ tục cho law_evidence cuối là việc
của kênh rule riêng trong luồng chính, KHÔNG thuộc file này.)

So bge-m3 thô: hybrid+legal_query bắt điều NỘI DUNG tốt hơn (bảng substantive-only:
Recall 15%->20%, HitRate 44%->52%) -> judge nhận đúng điều quyết định thắng-thua nhiều hơn.

CHẠY TRÊN KAGGLE (như eval_retrieval_bge.py):
  - Bật GPU. Kaggle Secrets: QDRANT_URL, QDRANT_KEY.
  - Upload: corpus_law_pub.json, retrieval_chunks_llm.json (bản legal_query).
    (Cho PRIVATE: đổi sang corpus private + retrieval_chunks_llm.json sinh từ case private.)
  - Dán cả file vào 1 cell rồi Run.
Đầu ra: retrieval_top10_ours.json  (dict {case_id: [N_SUB+N_PROC điều]}, đúng schema file mẫu).

DYNAMIC-K CHỌN LỌC: mỗi vụ = N_SUB điều nội dung (cho judge) + N_PROC điều thủ tục (cho
recall bằng chứng). Nộp >10 điều được, nhưng KHÔNG quăng hết (precision nát). Chỉnh N_SUB/
N_PROC ở đầu file. Mặc định 6+8=14 điều/vụ (F2 gần đỉnh trên public, robust).
"""
from __future__ import annotations

import glob
import json
import os
import re

MODEL_ID = "BAAI/bge-m3"
COLLECTION = "laws_bge_m3_v2_correct_pooling"
MAX_LENGTH = 8192
N_CAND = 50
# DYNAMIC-K CÓ CHỌN LỌC (được nộp >10 điều, nhưng KHÔNG quăng hết -> precision nát).
# Mỗi vụ = N_PROC điều THỦ TỤC (pool CORE tần suất cao, đứng đầu) + N_SUB điều NỘI DUNG
# (hybrid, top đầu). Đo F2 trên public: F2 chững ~30% quanh (N_SUB 5-6, N_PROC 8-11);
# thêm nữa chỉ tăng recall tí mà precision rớt -> KHÔNG đáng.
#   N_SUB=6  : đủ điều nội dung cho judge phán (điều quyết định thường nằm top-6).
#   N_PROC=8 : bắt phần lớn gold thủ tục, robust hơn 11 (top-8 CORE là điều phổ quát nhất;
#              4 điều đuôi tần suất thấp dễ overfit public). Muốn ép F2 public tối đa -> 11.
# Điều thủ tục SAI do bge-m3 nhầm (vd Điều 7 Tố tụng hành chính) vẫn bị loại; chỉ thủ tục
# ĐÚNG từ CORE được nhồi.
N_SUB = 6
N_PROC = 8

# Luật THỦ TỤC — LOẠI khỏi kênh nội dung (giống retrieve_law_substantive: `not is_procedural`).
# Chúng không giúp phán thắng-thua và được xử bởi kênh rule riêng -> không để lọt vào file này.
PROCEDURAL_LAW_IDS = {"92/2015/QH13", "93/2015/QH13", "326/2016/UBTVQH14", "26/2008/QH12"}

# Pool thủ tục (chỉ dùng nếu N_PROC>0) — xếp theo tần suất gold public giảm dần.
CORE_PROCEDURAL = [
    ("92/2015/QH13", 26), ("92/2015/QH13", 147), ("92/2015/QH13", 35),
    ("92/2015/QH13", 39), ("92/2015/QH13", 273), ("92/2015/QH13", 157),
    ("92/2015/QH13", 165), ("326/2016/UBTVQH14", 12), ("326/2016/UBTVQH14", 26),
    ("92/2015/QH13", 271), ("92/2015/QH13", 266),
]


def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ[name]


def _find(name):
    for p in [name, f"/kaggle/working/{name}"] + glob.glob(f"/kaggle/input/**/{name}", recursive=True):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Không thấy {name}")


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def main():
    import importlib.util
    import subprocess
    import sys
    for pkg, mod in [("qdrant-client", "qdrant_client"), ("sentence-transformers", "sentence_transformers"),
                     ("rank-bm25", "rank_bm25")]:
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

    import torch
    from qdrant_client import QdrantClient
    from sentence_transformers import SentenceTransformer

    assert torch.cuda.is_available(), "Bật GPU Kaggle."
    qc = QdrantClient(url=_get_secret("QDRANT_URL"), api_key=_get_secret("QDRANT_KEY"), timeout=300)

    corpus = load_json(_find("corpus_law_pub.json"))
    llm = load_json(_find("retrieval_chunks_llm.json"))
    llm_by_case = {c["case_id"]: c for c in llm["cases"]}

    # corpus_by_key: (law_id, article_no) -> {aid, content_Article}; corpus_records cho BM25.
    corpus_by_key, corpus_records = {}, []
    for law in corpus:
        for article_no, article in enumerate(law["content"], 1):
            key = (str(law["law_id"]), article_no)
            text = str(article.get("content_Article", ""))
            corpus_by_key[key] = {"aid": int(article["aid"]), "content_Article": text}
            corpus_records.append((key[0], key[1], int(article["aid"]), text))

    model = SentenceTransformer(MODEL_ID, device="cuda")
    model.max_seq_length = MAX_LENGTH

    def _tok(t):
        return re.findall(r"\w+", str(t).lower(), flags=re.UNICODE)

    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi([_tok(r[3]) for r in corpus_records])

    def _dense(query, limit):
        qv = model.encode([query or " "], normalize_embeddings=True, convert_to_numpy=True)[0]
        hits = qc.query_points(COLLECTION, query=qv.tolist(), limit=limit, with_payload=True).points
        return [(str(h.payload["law_id"]), int(h.payload["article_no"])) for h in hits]

    def _bm25(query, limit):
        scores = bm25.get_scores(_tok(query))
        order = sorted(range(len(scores)), key=lambda i: -scores[i])[:limit]
        return [(corpus_records[i][0], corpus_records[i][1]) for i in order]

    def _rrf(rankings, k=60):
        agg = {}
        for ranking in rankings:
            for rank, key in enumerate(ranking):
                agg[key] = agg.get(key, 0.0) + 1.0 / (k + rank + 1)
        return sorted(agg.items(), key=lambda x: (-x[1], x[0]))

    proc_keys = [k for k in dict.fromkeys(CORE_PROCEDURAL) if k in corpus_by_key][:N_PROC]

    def _row(law_id, article_no, rank, score):
        meta = corpus_by_key[(law_id, article_no)]
        return {"rank": rank, "score": round(float(score), 6), "law_id": law_id,
                "aid": meta["aid"], "article_no": article_no,
                "content_Article": meta["content_Article"]}

    out = {}
    for case_id in llm_by_case:
        query = llm_by_case[case_id].get("retrieval_text", "")
        fused = _rrf([_dense(query, N_CAND), _bm25(query, N_CAND)])
        # NỘI DUNG: bỏ luật thủ tục (loại thủ tục SAI do bge-m3 nhầm), lấy top N_SUB.
        seen = set(proc_keys)
        subs = []
        for key, _ in fused:
            if key in seen or key not in corpus_by_key or key[0] in PROCEDURAL_LAW_IDS:
                continue
            seen.add(key)
            subs.append(key)
            if len(subs) >= N_SUB:
                break
        # nội dung TRƯỚC (judge thấy điều quyết định ngay) + thủ tục sau. KHÔNG cắt 10.
        ordered = subs + proc_keys
        score_of = dict(fused)
        items = []
        for rank, (law_id, article_no) in enumerate(ordered, 1):
            sc = score_of.get((law_id, article_no), 0.5)  # thủ tục nhồi không có score dense -> 0.5
            items.append(_row(law_id, article_no, rank, sc))
        out[case_id] = items
        print(f"  ✓ {case_id}: {len(items)} điều ({len(subs)} nội dung + {len(proc_keys)} thủ tục)",
              flush=True)

    with open("retrieval_top10_ours.json", "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    n_avg = sum(len(v) for v in out.values()) / len(out) if out else 0
    print(f"[saved] retrieval_top10_ours.json · {len(out)} vụ · "
          f"N_SUB={N_SUB} + N_PROC={N_PROC} = {N_SUB + N_PROC} điều/vụ (TB {n_avg:.1f})")


if __name__ == "__main__":
    main()


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  ✓ case_4101: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_4337: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_4588: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_4616: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_6616: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_5226: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_6284: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_3995: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_6384: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_8219: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_2705: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_3263: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_9125: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_5658: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_5045: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_4920: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_8758: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_9089: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_127: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_2978: 14 điều (6 nội dung + 8 thủ tục)
  ✓ case_4579: 14 điều (6 nội dung + 8 th